In [ ]:
from astropy.table import Table

In [ ]:
import matplotlib.pyplot as plt
import arya

In [ ]:
from astropy import units as u

In [ ]:
import numpy as np

In [ ]:
from pathlib import Path

In [ ]:
import sys
sys.path.append("..")
sys.path.append("../../imaging")

from phot_utils import xmatch, get_atm_extinction
from astropy.nddata import CCDData

In [ ]:
import tomllib

In [ ]:
objname = "yasone2"

In [ ]:
cat_original = Table.read(f"../{objname}/allcolours.cat")
cat_julen_sep = Table.read(f"../{objname}/julen.cat")

cat_julen_se = Table.read(f"../{objname}/allcolours_julen.cat")

In [ ]:
xmatch_idx, xmatch_dist, xmatch_filt = xmatch(cat_original["R_ALPHA_J2000"], cat_original["R_DELTA_J2000"], cat_julen["R_ALPHA_J2000"], cat_julen["R_DELTA_J2000"], 1*u.arcsec)

xmatch_idx_sep, xmatch_dist_sep, xmatch_filt_sep = xmatch(cat_original["R_ALPHA_J2000"], cat_original["R_DELTA_J2000"], cat_julen_sep["R_ALPHA_J2000"], cat_julen_sep["R_DELTA_J2000"], 1*u.arcsec)


In [ ]:
def weighted_median(values, weights):
    """
    Compute the weighted median of an array of values.
    
    This implementation sorts values and computes the cumulative
    sum of the weights. The weighted median is the smallest value for
    which the cumulative sum is greater than or equal to half of the
    total sum of weights.
    Parameters
    ----------
    values : array-like
        List or array of values on which to calculate the weighted median.
    weights : array-like
        List or array of weights corresponding to the values.
    Returns
    -------
    float
        The weighted median of the input values.
    """
    # Convert input values and weights to numpy arrays
    values = np.array(values)
    weights = np.array(weights)
    
    # Get the indices that would sort the array
    sort_indices = np.argsort(values)
    
    # Sort values and weights according to the sorted indices
    values_sorted = values[sort_indices]
    weights_sorted = weights[sort_indices]  

    # Compute the cumulative sum of the sorted weights
    cumsum = weights_sorted.cumsum()
    
    # Calculate the cutoff as half of the total weight sum
    cutoff = weights_sorted.sum() / 2.
    
    # Return the smallest value for which the cumulative sum is greater
    # than or equal to the cutoff
    return values_sorted[cumsum >= cutoff][0]

In [ ]:
for col in ["G_MAG", "R_MAG", "I_MAG"]:
    cat_original[f"{col}_Julen"] = np.ma.masked_array(cat_julen[col][xmatch_idx], ~xmatch_filt)

    cat_original[f"{col}_Julen_sep"] = np.ma.masked_array(cat_julen_sep[col][xmatch_idx_sep], ~xmatch_filt_sep)

In [ ]:
import astropy

In [ ]:
zeropoints = {}

for filtname in ["G", "R", "I"]:
    mag0 = cat[f"{filtname.upper()}_MAG"]
    mag_ps = cat[f"{filtname.upper()}_MAG_Julen"]
    filt = cat["R_FLAGS"] == 0
    filt &= ~mag0.mask
    filt &= mag_ps > -10

    w = 1/(cat["R_MAG_ERR"]**2)

    m_weighted = weighted_median(mag0[filt] - mag_ps[filt], w[filt])

    m = np.median(mag0[filt] - mag_ps[filt])
    m_sc, med_sc, err_sc = astropy.stats.sigma_clipped_stats(mag0[filt] - mag_ps[filt])
    print(m_weighted, m, m_sc, med_sc)
    zeropoints[filtname] = med_sc

In [ ]:
cat = cat_original
fig, axs = plt.subplots(2, 3, sharex=True, sharey="row", figsize=(7, 4), gridspec_kw={"hspace": 0, "wspace": 0})
c = None
for i in range(3):
    plt.sca(axs[0][i])
    cs = arya.COLORS[0] #np.log10(cat["FWHM_WORLD"].to("arcsec") / u.arcsec)

    filt = ["G", "R", "I"][i]
    mag0 = cat[f"{filt}_MAG"]
    mag_err = cat[f"{filt}_MAG_ERR"]
    mag_ps = cat[f"{filt}_MAG_Julen"] + zeropoints[filt]
    # mag_ps_err = cat[f"mag_{filt}_err_panstarrs"]
    plt.scatter(mag0, mag_ps, s=1,  c=cs, alpha=0.1)
    plt.xlim(27, 17)
    plt.ylim(27, 17)
    plt.plot([27, 10], [27, 10], color="black", zorder=-1)
    if i == 0:
        plt.ylabel(f"mag (PS)")
    
    plt.sca(axs[1][i])
    
    # yerr = np.maximum(mag_ps_err, 0)
    # plt.errorbar(mag0[1:-1:20], (mag_ps-mag0)[1:-1:20], yerr=yerr[1:-1:20], fmt=".", capsize=0, )
    c = plt.scatter(mag0, (mag_ps - mag0), s=1, c=cs, alpha=0.1)

    plt.ylim(-0.5, 0.5)
    plt.axhline(0, color="black", zorder=-1)
    
    plt.xlabel(f"{filt} (osiris)")
    if i == 0:
        plt.ylabel("residual")


plt.tight_layout()

In [ ]:
cat = cat_original
fig, axs = plt.subplots(2, 3, sharex=True, sharey="row", figsize=(7, 4), gridspec_kw={"hspace": 0, "wspace": 0})
c = None
for i in range(3):
    plt.sca(axs[0][i])
    cs = arya.COLORS[0] #np.log10(cat["FWHM_WORLD"].to("arcsec") / u.arcsec)

    filt = ["G", "R", "I"][i]
    mag0 = cat[f"{filt}_MAG"]
    mag_err = cat[f"{filt}_MAG_ERR"]
    mag_ps = cat[f"{filt}_MAG_Julen_sep"] + 0.2
    # mag_ps_err = cat[f"mag_{filt}_err_panstarrs"]
    plt.scatter(mag0, mag_ps, s=1,  c=cs, alpha=0.1)
    plt.xlim(27, 17)
    plt.ylim(27, 17)
    plt.plot([27, 10], [27, 10], color="black", zorder=-1)
    if i == 0:
        plt.ylabel(f"mag (PS)")
    
    plt.sca(axs[1][i])
    
    # yerr = np.maximum(mag_ps_err, 0)
    # plt.errorbar(mag0[1:-1:20], (mag_ps-mag0)[1:-1:20], yerr=yerr[1:-1:20], fmt=".", capsize=0, )
    c = plt.scatter(mag0, (mag_ps - mag0), s=1, c=cs, alpha=0.1)

    plt.ylim(-0.5, 0.5)
    plt.axhline(0, color="black", zorder=-1)
    
    plt.xlabel(f"{filt} (osiris)")
    if i == 0:
        plt.ylabel("residual")


plt.tight_layout()

In [ ]:
cat = cat_original
fig, axs = plt.subplots(2, 3, sharex=True, sharey="row", figsize=(7, 4), gridspec_kw={"hspace": 0, "wspace": 0})
c = None
for i in range(3):
    plt.sca(axs[0][i])
    cs = arya.COLORS[0] #np.log10(cat["FWHM_WORLD"].to("arcsec") / u.arcsec)

    filt = ["G", "R", "I"][i]
    mag0 = cat[f"{filt}_MAG_Julen"] + zeropoints[filt]
    mag_err = cat[f"{filt}_MAG_ERR"]
    mag_ps = cat[f"{filt}_MAG_Julen_sep"]
    # mag_ps_err = cat[f"mag_{filt}_err_panstarrs"]
    plt.scatter(mag0, mag_ps, s=1,  c=cs, alpha=0.1)
    plt.xlim(27, 17)
    plt.ylim(27, 17)
    plt.plot([27, 10], [27, 10], color="black", zorder=-1)
    if i == 0:
        plt.ylabel(f"mag (PS)")
    
    plt.sca(axs[1][i])
    
    # yerr = np.maximum(mag_ps_err, 0)
    # plt.errorbar(mag0[1:-1:20], (mag_ps-mag0)[1:-1:20], yerr=yerr[1:-1:20], fmt=".", capsize=0, )
    c = plt.scatter(mag0, (mag_ps - mag0), s=1, c=cs, alpha=0.1)

    plt.ylim(-0.5, 0.5)
    plt.axhline(0, color="black", zorder=-1)
    
    plt.xlabel(f"{filt} (osiris)")
    if i == 0:
        plt.ylabel("residual")


plt.tight_layout()